# 03 — Entrenar CNN Heatmap Pelota ② (custom encoder-decoder, sin librería)

Entrena `BallHeatmapNet` (definida en `src/models/cnn.py`). La red recibe un frame
y predice un mapa de calor con la posición de la pelota. Está pensada para complementar
al detector ③, que falla con la pelota por desbalance de clases.

Output: `weights/ball_heatmap.pt`

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd /content
!rm -rf volley-ai
!git clone https://github.com/Angelote567/DeepLearning.git volley-ai
%cd volley-ai
!pip install -q torch torchvision tqdm Pillow PyYAML
import os
os.makedirs('data', exist_ok=True)
if not os.path.exists('data/volleyball-2'):
    os.symlink('/content/drive/MyDrive/volley-ai/data/volleyball-2', 'data/volleyball-2')
if os.path.exists('weights') and not os.path.islink('weights'):
    if not os.listdir('weights') or os.listdir('weights') == ['.gitkeep']:
        import shutil; shutil.rmtree('weights')
if not os.path.exists('weights'):
    os.symlink('/content/drive/MyDrive/volley-ai/weights', 'weights')

In [ ]:
import torch
print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

In [ ]:
from src.train.train_cnn_ball import train
best = train(epochs=60, batch_size=8, lr=1e-3)
print('Mejores pesos:', best)

In [ ]:
# Visualización rápida — pintar el heatmap predicho sobre 4 imágenes de validación
import torch, glob, random
from PIL import Image
import torchvision.transforms.functional as TF
import matplotlib.pyplot as plt
from src.models.cnn import BallHeatmapNet, heatmap_to_point
from src.config import HEATMAP_SIZE

model = BallHeatmapNet().cuda().eval()
ckpt = torch.load('weights/ball_heatmap.pt', map_location='cuda')
model.load_state_dict(ckpt['model'])

test_dir = 'data/volleyball-2/valid/images'
samples = random.sample(glob.glob(f'{test_dir}/*'), 4)
fig, axs = plt.subplots(2, 4, figsize=(16, 8))
for i, path in enumerate(samples):
    img = Image.open(path).convert('RGB').resize((HEATMAP_SIZE, HEATMAP_SIZE))
    x = TF.to_tensor(img)
    x = torch.cat([x, x, x], dim=0).unsqueeze(0).cuda()
    heat = model(x)[0, 0].cpu().numpy()
    pt = heatmap_to_point(torch.from_numpy(heat))
    axs[0, i].imshow(img); axs[0, i].set_title(f'pred={pt}'); axs[0, i].axis('off')
    axs[1, i].imshow(heat, cmap='hot'); axs[1, i].axis('off')
plt.tight_layout(); plt.show()